<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/09.transformer_1.vocab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [11]:
# Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
import os

repository_name = 'nlp2026'
repository_url = f'https://github.com/kikim6114/{repository_name}.git'

# 항상 루트 경로(/content/)로 이동 후 확인
%cd /content/

if not os.path.exists(repository_name):
    !git clone {repository_url}
    print(f"{repository_name} 클론 완료")
else:
    print(f"{repository_name} 폴더가 이미 존재합니다. 클론을 건너뜁니다.")
%cd nlp2026

/content
nlp2026 폴더가 이미 존재합니다. 클론을 건너뜁니다.
/content/nlp2026


# Tokenizer 훈련 및 Vocabulary 준비
- 한국어 위키백과 코퍼스 사용

## 1. 한국어 위키백과 Dataset 준비

### 사전 처리된 kowiki.txt 내려받기:
- 신속한 실습 진행을 위해, 사전에 처리해 놓은 데이터셋을 내려받아 사용하기로 한다
- 여기에서 제공되는 링크는 2026년 1학기 동안만 존재한다.
- 따라서, 그 이후에는 아래의 `kowiki.txt` 만들기 코드를 실행해서 직접 변환해야 할 것이다.

In [ ]:
!pip install --upgrade gdown
!pip install sentencepiece

In [12]:
# 필요한 패키지 import
import pandas as pd
import sentencepiece as spm
import torch
from torch import nn
import sys
import csv
import os
import gdown

`kowiki.txt` 다운로드

In [ ]:
dataset_folder = '1h58VFFL0ACe_jNQzTr-LeCYl_wS6D5w6'
dataset_url = f'https://drive.google.com/drive/folders/{dataset_folder}'

gdown.download_folder(dataset_url, quiet=False, use_cookies=False)

Retrieving folder contents


Retrieving folder 1XsDsXhCxUHk9xW1ftJbreRlkVJF20cBd kowiki
Processing file 1BLuq4mwvvydg_p2sEq-dEOp9QH2LWqdJ kowiki.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1BLuq4mwvvydg_p2sEq-dEOp9QH2LWqdJ
From (redirected): https://drive.google.com/uc?id=1BLuq4mwvvydg_p2sEq-dEOp9QH2LWqdJ&confirm=t&uuid=87487bf5-8bdc-4461-954b-a476707dba27
To: /content/nlp2026/nlp2026-data/kowiki/kowiki.txt
100%|██████████| 971M/971M [00:14<00:00, 65.5MB/s]
Download completed


['/content/nlp2026/nlp2026-data/kowiki/kowiki.txt']

kowiki 폴더 위치:
- Colab : ./nlp2026-data/kowiki
- Local: ./kowiki

In [ ]:
%ls /content/nlp2026/nlp2026-data/kowiki

'새 폴더'/   kowiki.model   kowiki.txt   kowiki.vocab


### 한국어 위키백과 Dataset 다운로드 및 변환 (위에서 `kowiki.txt` 내려받기에 성공했으면 다음 Cell은 SKIP!)

한국어 위키백과 최신 dump 파일을 `CSV` 파일 형식으로 내려받는 법
- `github.com/paul-hyun/web-crawler`에서 `README.md`의 안내에 따라 다운로드 받는다.
- `python kowiki.py` 실행 시 **kowiki 폴더 위치**가 앞의 cell에서 사용된 폴더 위치와 일치하도록 수정해야 한다.
- kowiki 폴더아래 kowiki_yyyymmdd.csv 형태의 파일이 생성됨
- [NOTE] [위키백과: 데이터베이스 다운로드 ⟹ 한국어 위키백과 dump 파일의 종류](https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B2%A0%EC%9D%B4%EC%8A%A4_%EB%8B%A4%EC%9A%B4%EB%A1%9C%EB%93%9C)

In [ ]:
# kowiki.txt 내려받았으면 이 Cell은 SKIP!!

# sys.maxsize = 9223372036854775807 ==> C의 long int 보다 커서 error 발생
csv.field_size_limit(170000)
in_file = "./kowiki/kowiki_20250506.csv"    # 현재 환경에 맞춰 수정할 것
out_file = "./kowiki/kowiki.txt"            # 현재 환경에 맞춰 수정할 것

if not os.path.exists(out_file):
    SEPARATOR = u"\u241D"  # Symbol For Group Separator ␝
    df = pd.read_csv(in_file, sep=SEPARATOR, engine="python")
    print(df.head())
    with open(out_file, "w", encoding='UTF8') as f:
        for index, row in df.iterrows():
            f.write(row["text"]) # title 과 text가 중복 되므로 text만 저장
            f.write("\n\n\n\n")  # 구분
    print(f"Creation of {out_file} is done.")
else:
    print(f"{out_file} already exists.")

./kowiki/kowiki.txt already exists.


## 2. Vocabulary 만들기

- `vocab_size` : ETRI KorBERT는 32,000개 SKT KoBERT는 8,000개를 사용.
- vocab_size가 커지면 성능이 좋아 지고 모델 파라미터 수가 증가한다.
- vocab 만들기가 성공적으로 끝나면, `kowiki` 폴더 아래에 `kowiki.model`과 `kowiki.vocab`이 생성된다.

#### 10분 이상의 시간이 걸릴 수 있으므로, 이미 만들어 내려받은 파일을 사용한다.
- Vocab 테스트로 SKIP 한다.

In [ ]:
# 실습 중에는 SKIP!
%%time
corpus = "./nlp2026-data/kowiki/kowiki.txt"   # Local: "./kowiki/kowiki.txt"
prefix = "./nlp2026-data/kowiki/kowiki"       # Local: "./kowiki/kowiki"
vocab_size = 8000
spm.SentencePieceTrainer.train(
    f"--input={corpus} --model_prefix={prefix} --vocab_size={vocab_size + 7}" +
    " --model_type=bpe" +
    " --max_sentence_length=999999" + # 문장 최대 길이
    " --pad_id=0 --pad_piece=[PAD]" + # pad (0)
    " --unk_id=1 --unk_piece=[UNK]" + # unknown (1)
    " --bos_id=2 --bos_piece=[BOS]" + # begin of sequence (2)
    " --eos_id=3 --eos_piece=[EOS]" + # end of sequence (3)
    " --user_defined_symbols=[SEP],[CLS],[MASK]") # 사용자 정의 토큰

#### Vocab 테스트

In [13]:
vocab_file = "./nlp2026-data/kowiki/kowiki.model"   #  Local: "./kowiki/kowiki.model"
vocab = spm.SentencePieceProcessor()
vocab.load(vocab_file)

lines = [
    "자연어처리에 관심이 많지만 좀 어렵다고 느껴집니다.",
    "너무 발전 속도가 빨라서 따라가기 벅차요.",
    "어떻게 하면 자연어처리 테크닉에 능숙할 수 있을까요?"
    ]

# text를 tensor로 변환
inputs = []
for line in lines:
    pieces = vocab.encode_as_pieces(line)
    ids = vocab.encode_as_ids(line)
    inputs.append(torch.tensor(ids))
    print(line)
    print(pieces)
    print(ids)
    print()

자연어처리에 관심이 많지만 좀 어렵다고 느껴집니다.
['▁자연', '어', '처', '리에', '▁관심', '이', '▁많', '지만', '▁좀', '▁어', '렵', '다고', '▁느', '껴', '집', '니다', '.']
[1145, 3754, 3969, 559, 3638, 3717, 210, 99, 3642, 137, 4465, 658, 1218, 5054, 3999, 1356, 3719]

너무 발전 속도가 빨라서 따라가기 벅차요.
['▁너무', '▁발전', '▁속', '도가', '▁빨', '라', '서', '▁따라', '가', '기', '▁', '벅', '차', '요', '.']
[2796, 1046, 293, 893, 2903, 3753, 3732, 264, 3728, 3735, 3716, 5160, 3872, 3886, 3719]

어떻게 하면 자연어처리 테크닉에 능숙할 수 있을까요?
['▁어떻게', '▁하', '면', '▁자연', '어', '처', '리', '▁테', '크', '닉', '에', '▁능', '숙', '할', '▁수', '▁있을', '까', '요', '?']
[3372, 29, 3833, 1145, 3754, 3969, 3738, 519, 3863, 4452, 3720, 1049, 4279, 3882, 18, 1413, 3925, 3886, 4406]



In [ ]:
len(vocab)

8007